In [ ]:
# import necessary libraries
import os
import pandas as pd
import scanpy as sc
from scipy import sparse

In [ ]:
# configure
EXPR_PATH = '../data/raw/exprMatrix.tsv.gzip'
META_PATH = '../data/raw/meta.tsv'
PRE_MAPPING_FN = "../data/intermediate/asd_brain.h5ad"
BACKUP_DIR = '../data/backup'

In [ ]:
# load exprMatrix file in chunks to avoid storage and disk crash
try:
    if 'adata' in globals():
        print('Using existing adata in memory. shape =', getattr(adata, 'shape', None))
    else:
        if os.path.exists(PRE_MAPPING_FN):
            print('Loading pre-mapping AnnData from', PRE_MAPPING_FN)
            adata = sc.read_h5ad(PRE_MAPPING_FN)
            print('Loaded adata shape:', adata.shape)
        else:
            # Need to build AnnData from expr + meta
            if not os.path.exists(EXPR_PATH):
                raise FileNotFoundError(f'Expression file not found: {EXPR_PATH}')
            if not os.path.exists(META_PATH):
                raise FileNotFoundError(f'Metadata file not found: {META_PATH}')

            print('Reading metadata from', META_PATH)
            meta = pd.read_csv(META_PATH, sep='\t', index_col=0)
            print('meta shape:', meta.shape)

            print('Reading expression matrix in chunks...')
            chunksize = 2000  # genes per chunk
            it = pd.read_csv(EXPR_PATH, sep='\t', index_col=0, compression='gzip', iterator=True, chunksize=chunksize)
            mat = None
            genes = []
            cells = None
            n_chunks = 0
            for chunk in it:
                n_chunks += 1
                # ensure dtype float32 for memory
                print(f"  Chunk {n_chunks}, shape={chunk.shape}")
                chunk = chunk.astype('float32')
                if n_chunks == 1:
                    cells = chunk.columns.astype(str).tolist()
                genes.extend(chunk.index.astype(str).tolist())
                chunk_sparse = sparse.csr_matrix(chunk.values)
                mat = chunk_sparse if mat is None else sparse.vstack([mat, chunk_sparse], format='csr')
                if n_chunks % 10 == 0:
                    print(f'  loaded ~{len(genes)} genes...')
            print('Finished reading expression. Genes:', len(genes), 'Cells:', len(cells))

            # AnnData expects observations=cells, variables=genes
            X = mat.transpose().tocsr()  # shape (n_cells, n_genes)
            obs = pd.DataFrame(index=cells)
            var = pd.DataFrame(index=genes)
            adata = sc.AnnData(X=X, obs=obs, var=var)

            # Align metadata
            common = adata.obs.index.intersection(meta.index)
            if len(common) == 0:
                raise RuntimeError('No overlapping cell IDs between expression columns and meta.tsv index')
            if len(common) < adata.n_obs:
                print(f'Warning: only {len(common)} / {adata.n_obs} cells found in metadata. Subsetting to intersection.')
                adata = adata[common, :].copy()
            meta = meta.reindex(adata.obs.index)
            adata.obs = meta

            # Save pre-mapping snapshot
            save_snapshot(adata, PRE_MAPPING_FN)

    print('Step A DONE. Current adata shape:', adata.shape)
except Exception as e:
    # try to save any partial adata if present
    print('Error in loading/building AnnData:', e)
    if 'adata' in locals():
        err_fn = os.path.join(BACKUP_DIR, 'adata_error_snapshot_pre_mapping.h5ad')
        save_snapshot(adata, err_fn)
    raise